# Azure AI Agents with Model Context Protocol (MCP) Support - Python

This notebook demonstrates how to use Microsoft Foundry agents with Model Context Protocol (MCP) tools in Python. It shows how to create an intelligent agent that can leverage external MCP servers (like Microsoft Learn) for enhanced capabilities using keyless authentication.

The code targets `azure-ai-projects` 2.x, which creates agents with `project_client.agents.create_version(...)` and runs them through the OpenAI-compatible Responses API. The 1.x `azure-ai-agents` API (`McpTool`, threads, runs, `ToolApproval`) is not used here.

## Install Required Python Packages

First, we need to install the necessary Python packages:
- **azure-ai-projects** (2.x): Microsoft Foundry project client, including agent creation
- **azure-identity**: Provides keyless authentication using DefaultAzureCredential
- **openai**: OpenAI-compatible Responses and Conversations API used to run the agent (also a dependency of `azure-ai-projects`)

The MCP server runs remotely and Foundry calls it for you, so this notebook does not need the `mcp` package.

## Keyless Authentication Benefits

This notebook demonstrates **keyless authentication** which provides several advantages:
- ✅ **No API keys to manage** - Uses Azure identity-based authentication
- ✅ **Enhanced security** - No secrets stored in code or configuration files
- ✅ **Automatic credential rotation** - Azure handles credential lifecycle management
- ✅ **Role-based access control** - Uses Azure RBAC for fine-grained permissions
- ✅ **Multi-environment support** - Works seamlessly across development and production

The `DefaultAzureCredential` automatically selects the best available credential source:
1. **Managed Identity** (when running in Azure)
2. **Azure CLI** credentials (during local development)
3. **Visual Studio** credentials
4. **Environment variables** (if configured)
5. **Interactive browser** authentication (as fallback)

## Keyless Authentication Setup

**Prerequisites for keyless authentication:**

### For Local Development:
```bash
# Install Azure CLI and login
az login
# Verify your identity
az account show
```

### For Azure Environments:
- Enable **System-assigned Managed Identity** on your Azure resource
- Assign appropriate **RBAC roles** to the managed identity:
  - `Cognitive Services OpenAI User` for Azure OpenAI access
  - `AI Developer` for Azure AI Projects access

### Environment Variables (Optional):
```python
# These are automatically detected by DefaultAzureCredential
# AZURE_CLIENT_ID=<your-client-id>
# AZURE_CLIENT_SECRET=<your-client-secret>
# AZURE_TENANT_ID=<your-tenant-id>
```

**No API keys or connection strings needed!** 🔐

In [ ]:
%pip install "azure-ai-projects>=2.10.0,<3" "azure-identity>=1.27.0,<2" "openai>=3.44.0,<4"

## Import Required Libraries

Import the necessary Python modules:
- **os**: Read configuration from environment variables
- **AIProjectClient**: Main client for Microsoft Foundry projects
- **MCPTool, PromptAgentDefinition**: Describe the MCP server and the agent that uses it
- **DefaultAzureCredential**: Keyless authentication for Azure services
- **McpApprovalResponse, ResponseInputParam**: Approve MCP tool calls the agent wants to make

In [ ]:
import os

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from azure.identity import DefaultAzureCredential
from openai.types.responses.response_input_param import McpApprovalResponse, ResponseInputParam

## Configure Project and MCP Server Settings

Set up the configuration using environment variables with fallback defaults:
- **FOUNDRY_PROJECT_ENDPOINT**: Your Microsoft Foundry project endpoint
- **FOUNDRY_MODEL_NAME**: The name of a model deployment in that project
- **MCP_SERVER_URL**: The URL of the MCP server (defaults to Microsoft Learn API)
- **MCP_SERVER_LABEL**: A label to identify the MCP server (defaults to "mslearn")

This approach allows for flexible configuration across different environments.

In [ ]:
project_endpoint = os.environ.get(
    "FOUNDRY_PROJECT_ENDPOINT",
    "https://your-resource.services.ai.azure.com/api/projects/your-project",
)
model_deployment_name = os.environ.get("FOUNDRY_MODEL_NAME", "Your model deployment name")
mcp_server_url = os.environ.get("MCP_SERVER_URL", "https://learn.microsoft.com/api/mcp")
mcp_server_label = os.environ.get("MCP_SERVER_LABEL", "mslearn")

## Create Azure AI Project Client (Keyless Authentication)

Initialize the Azure AI Project client using **keyless authentication**:
- **endpoint**: The Azure AI Foundry project endpoint URL
- **credential**: Uses `DefaultAzureCredential()` for secure, keyless authentication
- **No API keys required**: Automatically discovers and uses the best available credential

**Authentication Flow:**
1. Checks for Managed Identity (in Azure environments)
2. Falls back to Azure CLI credentials (for local development)
3. Uses other available credential sources as needed

This approach eliminates the need to manage API keys or connection strings in your code.

In [ ]:
credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)

## Create MCP Tool Definition

Create an MCP tool that connects to the Microsoft Learn MCP server:
- **server_label**: Identifier for the MCP server
- **server_url**: URL endpoint of the MCP server
- **require_approval**: `"always"` makes the agent pause for approval before each MCP tool call; `"never"` runs them automatically
- **allowed_tools**: Optional list to restrict which tools can be used (omit it to allow all tools)

This tool will enable the agent to access Microsoft Learn documentation and resources.

In [ ]:
mcp_tool = MCPTool(
    server_label=mcp_server_label,
    server_url=mcp_server_url,
    require_approval="always",  # Every MCP tool call waits for your approval
    # allowed_tools=["microsoft_docs_search"],  # Optional: restrict which MCP tools the agent may call
)

## Create Agent and Execute Conversation (Keyless Workflow)

This section demonstrates the complete **keyless agent workflow**:

1. **Create AI Agent**: Create an agent version with your model deployment and the MCP tool
2. **Create Conversation**: Establish a conversation that keeps context across responses
3. **Send Message**: Ask the agent about Azure OpenAI vs OpenAI differences
4. **Handle Tool Approvals**: Approve each `mcp_approval_request` the agent returns and send the approvals back
5. **Display Results**: Show the MCP tool calls and the agent's answer
6. **Clean Up**: Delete the agent version created by this notebook

**Keyless Features:**
- ✅ **No hardcoded secrets** - All authentication handled by Azure identity
- ✅ **Secure by default** - Uses role-based access control
- ✅ **Simplified deployment** - No credential management required
- ✅ **Audit-friendly** - All access is tracked through Azure identity

The agent will use MCP tools to access Microsoft Learn resources with full security and no API key management.

In [ ]:
with project_client, project_client.get_openai_client() as openai_client:
    # Create (a new version of) the agent with the MCP tool
    agent = project_client.agents.create_version(
        agent_name="my-mcp-agent",
        definition=PromptAgentDefinition(
            model=model_deployment_name,
            instructions="You are a helpful agent that can use MCP tools to assist users. Use the available MCP tools to answer questions and perform tasks.",
            tools=[mcp_tool],
        ),
    )
    print(f"Created agent {agent.name}, version {agent.version}")
    agent_reference = {"agent_reference": {"name": agent.name, "type": "agent_reference"}}

    try:
        # A conversation keeps the context across several responses
        conversation = openai_client.conversations.create()
        print(f"Created conversation, ID: {conversation.id}")

        response = openai_client.responses.create(
            conversation=conversation.id,
            input="What's the difference between Azure OpenAI and OpenAI?",
            extra_body=agent_reference,
        )

        # Approve MCP tool calls until the agent stops asking for approval
        while True:
            approvals: ResponseInputParam = []
            for item in response.output:
                if item.type == "mcp_approval_request" and item.server_label == mcp_server_label:
                    print(f"Approving MCP tool call: {item.name}({item.arguments})")
                    approvals.append(
                        McpApprovalResponse(
                            type="mcp_approval_response",
                            approve=True,
                            approval_request_id=item.id,
                        )
                    )
            if not approvals:
                break
            response = openai_client.responses.create(
                conversation=conversation.id,
                input=approvals,
                extra_body=agent_reference,
            )

        # Show which MCP tools the agent called
        for item in response.output:
            if item.type == "mcp_call":
                print(f"MCP call: {item.server_label}.{item.name} -> {'error' if item.error else 'ok'}")

        print("\nAgent response:")
        print("-" * 50)
        print(response.output_text)
    finally:
        # Clean up the agent version created by this notebook
        project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
        print("Deleted agent version")